# Sparse & Efficient: MoE + MLA

Advanced efficiency techniques for trillion-parameter models. Learn how Mixture of Experts routes computation sparsely, and how Multi-head Latent Attention compresses KV tensors. Deep dive into DeepSeek V3 and Mixtral architectures.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/moe-mla)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Mixture of Experts (MoE)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """Single expert: FFN."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff)
        self.w2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.w2(F.relu(self.w1(x)))

class MoELayer(nn.Module):
    """Mixture of Experts with top-k routing."""
    def __init__(self, d_model, num_experts, expert_dim, top_k, load_balance_loss_weight=0.01):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.load_balance_loss_weight = load_balance_loss_weight

        # Create experts
        self.experts = nn.ModuleList([
            Expert(d_model, expert_dim) for _ in range(num_experts)
        ])

        # Router network
        self.router = nn.Linear(d_model, num_experts)

        self.load_balance_loss = 0.0

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)

        Returns:
            output: (batch, seq_len, d_model)
        """
        batch, seq_len, d_model = x.shape

        # Compute router logits: (batch, seq_len, num_experts)
        router_logits = self.router(x)

        # Get top-k experts: (batch, seq_len, top_k)
        router_probs = F.softmax(router_logits, dim=-1)
        top_k_probs, top_k_indices = torch.topk(router_probs, self.top_k, dim=-1)

        # Normalize top-k probabilities
        top_k_probs = top_k_probs / top_k_probs.sum(dim=-1, keepdim=True)

        # Initialize output
        output = torch.zeros_like(x)

        # Route tokens to experts
        for i in range(self.top_k):
            # Get expert indices for this position in top-k
            expert_idx = top_k_indices[:, :, i]  # (batch, seq_len)
            expert_probs = top_k_probs[:, :, i]  # (batch, seq_len)

            # Process each token through its assigned expert
            for expert_id in range(self.num_experts):
                mask = (expert_idx == expert_id)
                if mask.any():
                    # Get tokens routed to this expert
                    masked_x = x[mask]  # (n_tokens, d_model)
                    expert_output = self.experts[expert_id](masked_x)
                    # Add weighted output
                    output[mask] += expert_output * expert_probs[mask].unsqueeze(-1)

        # Compute load balancing loss
        # Encourage balanced expert usage
        expert_counts = torch.sum(router_probs > 0, dim=(0, 1))
        load_balance_loss = torch.var(expert_counts.float())
        self.load_balance_loss = self.load_balance_loss_weight * load_balance_loss

        return output

# Test MoE layer
d_model = 256
num_experts = 8
expert_dim = 512
top_k = 2

moe = MoELayer(d_model, num_experts, expert_dim, top_k)

x = torch.randn(4, 10, d_model)  # batch=4, seq_len=10
output = moe(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Load balance loss: {moe.load_balance_loss.item():.6f}")

# Count parameters
total_params = sum(p.numel() for p in moe.experts) + sum(p.numel() for p in moe.router.parameters())
active_params = (top_k / num_experts) * sum(p.numel() for p in moe.experts[0].parameters()) * num_experts + sum(p.numel() for p in moe.router.parameters())
print(f"Total experts params: {sum(p.numel() for p in moe.experts) / 1e6:.2f}M")
print(f"Active per token (top-k={top_k}): {(top_k / num_experts):.1%}")


### Lesson example: Multi-head Latent Attention (MLA)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadLatentAttention(nn.Module):
    """
    Multi-head Latent Attention: compress K/V to low-rank latent.
    """
    def __init__(self, d_model, n_heads, d_latent=128):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.d_latent = d_latent

        # Q projection (full)
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)

        # K/V projections (to latent)
        self.k_latent_proj = nn.Linear(d_model, d_latent)
        self.v_latent_proj = nn.Linear(d_model, d_latent)

        # Expand latent back to full K/V
        self.k_expand = nn.Linear(d_latent, n_heads * self.head_dim)
        self.v_expand = nn.Linear(d_latent, n_heads * self.head_dim)

        # Output projection
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, mask=None, cache_latent=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: (seq_len, seq_len) causal mask
            cache_latent: (batch, cached_len, d_latent) for KV caching

        Returns:
            output: (batch, seq_len, d_model)
            latent_kv: (batch, seq_len, d_latent) for caching
        """
        batch, seq_len, d_model = x.shape

        # Project Q (full attention heads)
        q = self.q_proj(x)  # (batch, seq_len, n_heads * head_dim)
        q = q.view(batch, seq_len, self.n_heads, self.head_dim)

        # Compress K/V to latent
        k_latent = self.k_latent_proj(x)  # (batch, seq_len, d_latent)
        v_latent = self.v_latent_proj(x)  # (batch, seq_len, d_latent)

        # Cache: optionally use cached K/V latent from previous tokens
        if cache_latent is not None:
            k_latent = torch.cat([cache_latent, k_latent], dim=1)
            v_latent = torch.cat([cache_latent, v_latent], dim=1)

        # Expand latent back to full K/V
        k = self.k_expand(k_latent)  # (batch, cached_seq + seq_len, n_heads * head_dim)
        v = self.v_expand(v_latent)  # (batch, cached_seq + seq_len, n_heads * head_dim)

        k = k.view(batch, -1, self.n_heads, self.head_dim)
        v = v.view(batch, -1, self.n_heads, self.head_dim)

        # Compute attention
        q_t = q.transpose(1, 2)  # (batch, n_heads, seq_len, head_dim)
        k_t = k.transpose(1, 2)  # (batch, n_heads, cached_seq + seq_len, head_dim)
        v_t = v.transpose(1, 2)

        scores = torch.matmul(q_t, k_t.transpose(-2, -1)) / (self.head_dim ** 0.5)

        # Apply mask
        if mask is not None:
            scores = scores + mask * -1e9

        attn_weights = F.softmax(scores, dim=-1)

        # Apply to values
        output = torch.matmul(attn_weights, v_t)  # (batch, n_heads, seq_len, head_dim)
        output = output.transpose(1, 2)  # (batch, seq_len, n_heads, head_dim)
        output = output.contiguous().view(batch, seq_len, -1)

        # Project output
        output = self.out_proj(output)

        # Return new latent for caching
        new_latent = k_latent if cache_latent is not None else k_latent[:, :seq_len]

        return output, new_latent

# Test MLA
d_model = 512
n_heads = 8
d_latent = 64  # 8x compression

mla = MultiHeadLatentAttention(d_model, n_heads, d_latent)

x = torch.randn(2, 10, d_model)

# Training (no caching)
output, latent = mla(x)
print(f"Output shape: {output.shape}")
print(f"Latent shape for caching: {latent.shape}")
print(f"Cache compression: {latent.numel() / (n_heads * d_model):.1%} of full KV")

# Generation with caching
cache_latent = latent[:, :5]  # Pretend we cached first 5 tokens
new_token = torch.randn(2, 1, d_model)
output_new, new_latent = mla(new_token, cache_latent=cache_latent)
print(f"Generation output: {output_new.shape}")
print(f"Cache size: {cache_latent.numel()} elements (latent) vs {cache_latent.numel() * (n_heads * d_model // d_latent)} (full KV)")


---

## Exercise: Build a Simple MoE Layer


Implement a basic MoE layer with top-k routing and load balancing loss. Create 4-8 experts, route tokens using softmax + top-k, and add an auxiliary loss to encourage balanced expert usage. Test with example inputs and verify parameter efficiency.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """Single expert FFN."""
    def __init__(self, d_model, d_ff):
        super().__init__()
        # TODO: Define expert layers (e.g., Linear -> GELU -> Linear)
        pass

    def forward(self, x):
        # TODO: Forward pass through expert
        pass

class SimpleMoE(nn.Module):
    """Simple Mixture of Experts with top-k routing."""
    def __init__(self, d_model, num_experts, d_ff, top_k):
        super().__init__()
        # TODO: Initialize experts, router, and parameters
        self.num_experts = num_experts
        self.top_k = top_k
        self.load_balance_loss = 0.0

    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)

        Returns:
            output: (batch, seq_len, d_model)
        """
        # TODO: Implement MoE routing
        # 1. Compute router logits
        # 2. Get top-k experts
        # 3. Route tokens to experts
        # 4. Compute load balancing loss
        # 5. Return output
        pass

if __name__ == '__main__':
    d_model = 256
    num_experts = 4
    d_ff = 512
    top_k = 2

    moe = SimpleMoE(d_model, num_experts, d_ff, top_k)
    x = torch.randn(4, 8, d_model)

    output = moe(x)
    print(f"Output shape: {output.shape}")
    print(f"Load balance loss: {moe.load_balance_loss:.6f}")

    # Verify parameters
    total_params = sum(p.numel() for p in moe.parameters())
    print(f"Total parameters: {total_params / 1e6:.2f}M")
    print(f"Active per token: {(top_k / num_experts):.1%}")
    print("✓ MoE test passed!")

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/moe-mla) for the solution and discussion.
